# 04 · Архивы RuASD на CPU
Выберите тип среды **«Без ускорителя»**. Блокнот использует тот же каталог и разбиение TAR, что и notebook 03. По умолчанию выбраны train и validation, всего ожидается 32 868 FLAC.

In [ ]:
# Репозиторий, лёгкие зависимости и Google Drive
import hashlib
import json
import subprocess
import sys
from pathlib import Path

from google.colab import drive, output
from IPython.display import display

repo = Path('/content/audio-deepfake-detector')
if not (repo / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/Unvand/audio-deepfake-detector.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'soundfile', 'tqdm', 'ipywidgets'],
    check=True,
)
sys.path.insert(0, str(repo))
drive.mount('/content/drive')
output.enable_custom_widget_manager()

In [ ]:
# Настройки проверки и архивации
import ipywidgets as widgets
import pandas as pd

from audio_deepfake_detector.src.colab_tar_cache import GIB, build_tar_archives

root = Path('/content/drive/MyDrive/VKR_RuASD')
source_root = root / 'prepared_v1'
archive_root = root / 'prepared_v1_tar'
staging_root = Path('/content/ruasd_tar_work')
report_path = root / 'tar_archives_report.json'

split_widgets = {
    split: widgets.Checkbox(description=split, value=split != 'test_unseen')
    for split in ('train', 'validation', 'test_unseen')
}
quota = widgets.FloatText(description='Свободное место Google Drive, ГиБ', style={'description_width': 'initial'})
checksum = widgets.Checkbox(description='Проверять SHA-256', value=True)
panel_output = widgets.Output()

def options():
    if quota.value <= 0:
        raise ValueError('Введите текущий свободный объём Google Drive в ГиБ.')
    manifest = source_root / 'ruasd_segments_manifest.csv'
    version = hashlib.sha256(manifest.read_bytes()).hexdigest()
    if not (archive_root / version).exists() and any(archive_root.glob('*/*/plan.json')):
        raise ValueError('Есть планы для другого manifest. Проверьте путь к исходному CSV.')
    splits = tuple(split for split, widget in split_widgets.items() if widget.value)
    return {'splits': splits, 'archive_free_bytes': int(quota.value * GIB)}

def inspect_archives(*, verify_sha=False, verify_contents=False):
    return build_tar_archives(
        source_root, archive_root, staging_root, **options(), plan_only=True,
        verify_sha=verify_sha, verify_contents=verify_contents,
    )

def show_report(report):
    display(pd.Series({key: value for key, value in report.items() if key != 'archives'}))
    table = pd.DataFrame(report['archives'])
    display(table.rename(columns={
        'archive': 'Архив', 'split': 'Split', 'size_bytes': 'Размер, Б',
        'sha256': 'SHA-256', 'status': 'Статус', 'error': 'Ошибка',
    }))
    if not report['enough_space']:
        print('Недостаточно места для записи оставшихся TAR и резерва в 1 ГиБ.')

def panel_action(action):
    with panel_output:
        panel_output.clear_output(wait=True)
        try:
            action()
        except (OSError, ValueError) as error:
            print(error)

def show_space():
    report = inspect_archives(verify_sha=checksum.value)
    display(pd.Series({
        'Свободно на Drive, ГиБ': report['drive_free_bytes'] / GIB,
        'Нужно ещё на Drive, ГиБ': report['additional_drive_bytes'] / GIB,
        'Незавершённые части уже занимают, ГиБ': report['partial_bytes_on_drive'] / GIB,
        'Свободно локально, ГиБ': report['local_free_bytes'] / GIB,
        'Максимальный временный TAR, ГиБ': report['peak_local_tar_bytes'] / GIB,
        'Резерв, ГиБ': report['reserve_bytes'] / GIB,
        'Места достаточно': report['enough_space'],
    }))

buttons = []
for label, action in (
    ('Проверить существующие TAR', lambda: show_report(inspect_archives(verify_sha=checksum.value))),
    ('Показать план архивации', lambda: show_report(inspect_archives())),
    ('Проверить свободное место', show_space),
):
    button = widgets.Button(description=label, layout=widgets.Layout(width='240px'))
    button.on_click(lambda _, action=action: panel_action(action))
    buttons.append(button)
display(widgets.VBox([widgets.HBox(list(split_widgets.values())), quota, checksum, *buttons, panel_output]))

## Продолжить архивацию
Введите актуальный остаток квоты и проверьте план. Просмотр плана сверяет размеры и JSON. Проверка SHA-256 считывает готовые TAR целиком. Установите переключатель в `True` и запустите следующую ячейку через ▶. После отключения повторите запуск с прежним выбором split и обновлённой квотой. Для test_unseen нужен отдельный запуск с выбором только этого split.

In [ ]:
RUN_TAR_BUILD = False

if RUN_TAR_BUILD:
    build_tar_archives(
        source_root, archive_root, staging_root, **options(), verify_contents=True,
    )
    final_report = inspect_archives(verify_sha=True, verify_contents=True)
    final_report['complete'] = final_report['pending_archives'] == 0
    final_report['selected_splits'] = list(options()['splits'])
    temporary = report_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(final_report, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(report_path)
    show_report(final_report)
    print('Отчёт сохранён в', report_path)
    if not final_report['complete']:
        raise RuntimeError('Некоторые части не прошли итоговую проверку.')

## Восстановить кэш в notebook 03
Когда появится GPU, в notebook 03 выберите **«Готовые TAR»** и запустите ячейку с `RUN_CACHE_PREPARATION = True`. Для продолжения обучения используйте прежний `run_id` и Resume. Верните `RUN_TAR_BUILD` в `False` перед сохранением notebook 04.